# MNIST outlier detection with IMSTE

Fit a higher-dimensional IMSTE embedding, rank samples with Local Outlier Factor (LOF), and inspect the 100 highest-scoring images. A sample is also marked as a detected outlier when it has no reciprocal link among its `k` nearest embedding neighbors.

The balanced 3,000-image sample is deterministic. MNIST is loaded through the project cache and downloaded only on the first run.

In [11]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'imste').is_dir() and (path / 'notebooks' / 'src').is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError('Run this notebook from the repository or one of its subdirectories.')
sys.path.insert(0, str(repo_root / 'notebooks' / 'src'))
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from imste import IteratedMinimumSpanningTreeEmbedder
from high_dim_mst_gallery import load_mnist_family_dataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load a balanced sample

Keep raw pixels for the image panel and standardize a copy for IMSTE. Labels are used only to draw a balanced sample and label the images; they are not passed into outlier detection.

In [12]:
SAMPLES_PER_CLASS = 300
RANDOM_STATE = 42

X_all, y_all = load_mnist_family_dataset('MNIST')
rng = np.random.default_rng(RANDOM_STATE)
selected = np.concatenate([
    rng.choice(np.flatnonzero(y_all == label), SAMPLES_PER_CLASS, replace=False)
    for label in np.unique(y_all)
])
rng.shuffle(selected)
X_images = np.asarray(X_all[selected], dtype=np.float32)
y = np.asarray(y_all[selected])
X = StandardScaler().fit_transform(X_images).astype(np.float32)
print(f'Selected {len(X):,} images; class counts: {np.unique(y, return_counts=True)[1].tolist()}')

Selected 3,000 images; class counts: [300, 300, 300, 300, 300, 300, 300, 300, 300, 300]


## Fit IMSTE and score local outliers

`mapper.outliers()` scores the fitted estimator. Pass a list of dimensions to average LOF ranks across embeddings; samples that remain outliers across dimensions are promoted in the consensus set.

In [ ]:
K = 3
ESTIMATOR_PARAMS = {
    'n_components': 5,
    'n_msts': 10,
    'n_epochs': 00,
    'random_state': RANDOM_STATE,
    'device': 'auto',
}

mapper = IteratedMinimumSpanningTreeEmbedder(**ESTIMATOR_PARAMS)
embedding = mapper.fit_transform(X)
result = mapper.outliers(k=K, n_components=[5, 10, 30])
print(f'Compared dimensions: {result.components}')
print(f'Zero-reciprocal-neighbor outliers: {len(result.outlier_indices):,}')
print(f'Highest LOF score: {result.lof_scores[result.ranked_indices[0]]:.3f}')

## Inspect the 100 highest LOF scores

Each panel title shows average LOF rank and outlier frequency across dimensions. `OUTLIER` marks samples with zero reciprocal neighbors in every evaluated dimension. The panel ranks every sample by average LOF rank.

In [ ]:
top_indices = result.ranked_indices[:100]
outlier_set = set(result.outlier_indices.tolist())
image_side = int(round(np.sqrt(X_images.shape[1])))
if image_side * image_side != X_images.shape[1]:
    raise ValueError(f'Expected flattened square images, got {X_images.shape[1]} features.')

fig, axes = plt.subplots(10, 10, figsize=(13, 13))
for rank, (ax, sample_index) in enumerate(zip(axes.flat, top_indices), start=1):
    pixels = X_images[sample_index].reshape(image_side, image_side)
    ax.imshow(pixels, cmap='gray_r', vmin=0, vmax=255)
    flag = ' OUTLIER' if sample_index in outlier_set else ''
    ax.set_title(
        f"#{rank} y={y[sample_index]}\nAvg rank={result.average_ranks[sample_index]:.1f} "
        f"Outlier dims={result.outlier_frequency[sample_index]}/{len(result.components)}{flag}",
        fontsize=6, color='crimson' if flag else 'black'
    )
    ax.set_axis_off()
fig.suptitle('Top 100 MNIST LOF candidates', fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()

In [ ]:
import pandas as pd

pd.DataFrame({
    'sample_index': top_indices[:20],
    'digit': y[top_indices[:20]],
    'LOF score (first dimension)': result.lof_scores[top_indices[:20]],
    'average LOF rank': result.average_ranks[top_indices[:20]],
    'outlier dimensions': result.outlier_frequency[top_indices[:20]],
    'outlier in all dimensions': [i in outlier_set for i in top_indices[:20]],
})